In [2]:
pip install requests beautifulsoup4 pandas lxml

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install selenium webdriver-manager pandas

Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
import time
import random
import re

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


# Driver Setup
def start_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), 
        options=options
    )

driver = start_driver()

In [10]:
# Helper Functions

#To extract the price
def extract_price(text):
    match = re.search(r"€[\d,]+", text)
    return int(match.group().replace("€", "").replace(",", "")) if match else None

#To extract which part of dublin
def extract_area(text):
    match = re.search(r"(Dublin\s?\d+)", str(text))
    return match.group(1) if match else "Unknown"

#To get listing id (Unique number)
def extract_listing_id(url):
    match = re.search(r"/(\d+)$", url)
    return match.group(1) if match else "N/A"

#To fetch how many beds and bathroom in a house
def extract_beds_baths(text):
    beds = re.search(r"(\d+)\s*bed", text.lower())
    baths = re.search(r"(\d+)\s*bath", text.lower())

    return (
        int(beds.group(1)) if beds else None,
        int(baths.group(1)) if baths else None
    )

#To extract what type of house
def extract_property_type(title):
    title = str(title).lower()

    if "semi-detached" in title:
        return "Semi-Detached"
    elif "end of terrace" in title:
        return "End of Terrace"
    elif "terraced" in title:
        return "Terraced"
    elif "detached" in title:
        return "Detached"
    elif "bungalow" in title:
        return "Bungalow"
    elif "duplex" in title:
        return "Duplex"
    elif "studio" in title:
        return "Studio"
    elif "apartment" in title:
        return "Apartment"
    else:
        return "Other"

#To get eircode
def extract_eircode(text):
    match = re.search(r"\b[A-Z]\d{2}\s?[A-Z0-9]{4}\b", text)
    return match.group() if match else "Unknown"

#To find nearby transport available
def detect_transport(text):
    text = text.lower()
    return {
        "has_bus": 1 if "bus" in text else 0,
        "has_dart": 1 if "dart" in text else 0,
        "has_luas": 1 if "luas" in text else 0
    }

#to find distance from city center
def estimate_distance(area):
    mapping = {
        "Dublin 1": 1, 
        "Dublin 2": 1, 
        "Dublin 3": 4, 
        "Dublin 4": 5,
        "Dublin 6": 5, 
        "Dublin 8": 3, 
        "Dublin 9": 6, 
        "Dublin 10": 8,
        "Dublin 11": 7, 
        "Dublin 12": 6, 
        "Dublin 13": 10, 
        "Dublin 14": 7,
        "Dublin 15": 12, 
        "Dublin 16": 10, 
        "Dublin 18": 14
    }
    return mapping.get(area, 10)

#To differentiate based on location to city center
def distance_category(dist):
    if dist <= 3:
        return "City Centre"
    elif dist <= 8:
        return "Suburban"
    else:
        return "Outer"



In [4]:
# Scrapping from daft.ie
BASE_URL = "https://www.daft.ie/property-for-sale/dublin" #URL for webscrapping
NUM_PAGES = 50 #No. of pages to be scrapped

data = []
seen_links = set()

for page in range(1, NUM_PAGES + 1):
    print(f"\nScraping page {page}")

    url =f"{BASE_URL}?page={page}"
    print(url)

    driver.get(url)
    time.sleep(5)

    listings = driver.find_elements(By.XPATH, "//a[contains(@href,'/for-sale/')]")

    links = list(set([
        l.get_attribute("href")
        for l in listings
        if l.get_attribute("href") and "/for-sale/" in l.get_attribute("href")
    ]))

    # Remove duplicates across pages
    links = [l for l in links if l not in seen_links]
    seen_links.update(links)

    print(f"Found {len(links)} new listings")

    # Limit per page 
    links = links[:20]

    for link in links:
        try:
            driver.get(link)
            time.sleep(random.uniform(3, 5))

            page_text = driver.page_source
            page_text_lower = page_text.lower()

            # Title
            try:
                title = driver.find_element(By.TAG_NAME, "h1").text
            except:
                title = "N/A"

            # Extract data
            price = extract_price(page_text)
            beds, baths = extract_beds_baths(page_text_lower)
            area = extract_area(page_text)
            listing_id = extract_listing_id(link)
            #property_type = extract_property_type(title)
            property_type = extract_property_type(title + " " + page_text)
            eircode = extract_eircode(page_text)

            transport = detect_transport(page_text_lower)
            distance = estimate_distance(area)
            dist_category = distance_category(distance)

            data.append({
                "listing_id": listing_id,
                "title": title,
                "property_type": property_type,
                "price": price,
                "beds": beds,
                "baths": baths,
                "eircode": eircode,
                "area": area,
                "distance_from_city_km": distance,
                "distance_category": dist_category,
                "has_bus": transport["has_bus"],
                "has_dart": transport["has_dart"],
                "has_luas": transport["has_luas"],
                "url": link
            })

            #testing
            print(f"{listing_id} | {property_type} | €{price} | {area} | {eircode}")

        except Exception as e:
            print("Skipping listing:", e)

    time.sleep(3)

# Save data
df = pd.DataFrame(data)

# Clean dataset
df = df.dropna(subset=["price", "beds", "baths", "eircode"])

df.to_csv("Group6_Daft_Dataset.csv", index=False)

print("\nFINAL DATASET CREATED")
print(df.head())


Scraping page 1
https://www.daft.ie/property-for-sale/dublin?page=1
Found 20 new listings
6526302 | Semi-Detached | €695000 | Dublin 4 | A94EW01
6505756 | Apartment | €325000 | Dublin 8 | D08H348
6525931 | Detached | €395000 | Dublin 18 | D18K7R8
6521768 | Semi-Detached | €595000 | Dublin 18 | A96X3F1
6489592 | Semi-Detached | €860000 | Dublin 18 | D18R8X2
6498487 | Semi-Detached | €995000 | Dublin 4 | D04KH63


KeyboardInterrupt: 

In [11]:
# Scraping from myHome.ie
BASE_URL = "https://www.myhome.ie/residential/dublin/property-for-sale"
NUM_PAGES = 20

for page in range(1, NUM_PAGES + 1):
    print(f"\nScraping page {page}")

    url = f"{BASE_URL}?page={page}"
    print(url)
    driver.get(url)
    time.sleep(5)

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)

    listings = driver.find_elements(By.XPATH, "//a[contains(@href,'/residential/brochure/')]")

    links = list(set([
        l.get_attribute("href")
        for l in listings if l.get_attribute("href")
    ]))

    links = [l for l in links if l not in seen_links]
    seen_links.update(links)

    print(f"Found {len(links)} listings")

    for link in links[:20]:
        try:
            driver.get(link)
            time.sleep(random.uniform(3, 5))

            page_text = driver.page_source
            page_text_lower = page_text.lower()

            # Title
            try:
                title = driver.find_element(By.TAG_NAME, "h1").text
            except:
                title = "N/A"

            # Extract all attributes
            price = extract_price(page_text)
            beds, baths = extract_beds_baths(page_text_lower)
            area = extract_area(page_text)
            listing_id = extract_listing_id(link)
            property_type = extract_property_type(title + " " + page_text)
            eircode = extract_eircode(page_text)

            transport = detect_transport(page_text_lower)
            distance = estimate_distance(area)
            dist_category = distance_category(distance)

            data.append({
                "listing_id": listing_id,
                "title": title,
                "property_type": property_type,
                "price": price,
                "beds": beds,
                "baths": baths,
                "eircode": eircode,
                "area": area,
                "distance_from_city_km": distance,
                "distance_category": dist_category,
                "has_bus": transport["has_bus"],
                "has_dart": transport["has_dart"],
                "has_luas": transport["has_luas"],
                "url": link
            })

            print(f"{listing_id} | {property_type} | €{price} | {area} | {eircode}")

        except Exception as e:
            print("Skipping:", e)

    time.sleep(3)


driver.quit()


# Saving Data
df = pd.DataFrame(data)

if not df.empty and "price" in df.columns:
    df = df.dropna(subset=["price"])

df.to_csv("Group6_MyHome_Dataset.csv", index=False)

print("\nFINAL DATASET CREATED")
print(df.head())


Scraping page 1
https://www.myhome.ie/residential/dublin/property-for-sale?page=1
Found 20 listings
Unknown | €310000 | Apartment | D24XR52
Dublin 18 | €4500000 | Detached | D18 H1F3


KeyboardInterrupt: 

In [ ]:
df1 = pd.read_csv("Group6_Daft_Dataset.csv")
df2 = pd.read_csv("Group6_MyHome_Dataset.csv")

final_df = pd.concat([df1, df2], ignore_index=True)

final_df.to_csv("Group6_Dataset.csv", index=False)